# Claude Agent SDK — Agentic Applications (Colab Walkthrough)
**Claude Certified Architect · Topic 1 — Domain 1: Agentic Architecture & Orchestration**

This notebook is the hands-on companion to the slide deck. It walks through:
1. The agentic loop (`stop_reason`) — raw Messages API
2. Tool integration with the Agent SDK (in-process MCP)
3. Multi-agent orchestration & subagent delegation (`AgentDefinition`)
4. Lifecycle hooks (`PreToolUse` / `PostToolUse`)
5. A programmatic prerequisite gate (deterministic enforcement)

> Set your API key in the next cell. Cells that call the live API are marked **(LIVE)** — they cost tokens.

In [ ]:
# Setup — install SDKs
!pip -q install anthropic claude-agent-sdk
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # <-- paste your key (or use Colab secrets)
print("Key set:", bool(os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant")))

## 1. The agentic loop — drive on `stop_reason`
The single most-tested idea in Domain 1.
- `tool_use` → execute the tool, append the result as a `user` turn, loop again.
- `end_turn` → done.

**Anti-patterns to avoid:** parsing natural-language text to decide when to stop,
using an iteration cap as the *primary* stop, or checking for assistant text as a
completion signal.

In [ ]:
import json
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-sonnet-4-5"

TOOLS = [{
    "name": "get_weather",
    "description": "Get the CURRENT weather for a city. Input: a city name string.",
    "input_schema": {"type": "object",
                     "properties": {"city": {"type": "string"}},
                     "required": ["city"]},
}]

def get_weather(city):
    db = {"paris": "18C, light rain", "tokyo": "27C, clear"}
    return {"city": city, "conditions": db.get(city.lower(), "unknown")}

TOOL_IMPLS = {"get_weather": lambda a: get_weather(**a)}

def run_agent(prompt, max_safety_turns=10):
    messages = [{"role": "user", "content": prompt}]
    for _ in range(max_safety_turns):          # circuit breaker, NOT primary stop
        resp = client.messages.create(model=MODEL, max_tokens=1024,
                                      tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "end_turn":
            return "".join(b.text for b in resp.content if b.type == "text")
        if resp.stop_reason == "tool_use":
            results = []
            for b in resp.content:
                if b.type == "tool_use":
                    out = TOOL_IMPLS[b.name](b.input)
                    results.append({"type": "tool_result", "tool_use_id": b.id,
                                    "content": json.dumps(out)})
            messages.append({"role": "user", "content": results})
            continue
        return f"[stopped: {resp.stop_reason}]"
    return "[circuit breaker]"

# (LIVE)
print(run_agent("What's the weather in Paris right now?"))

## 2. Tool integration with the Agent SDK
Define tools with `@tool`, bundle with `create_sdk_mcp_server`, pass to `query()`.
Tool names become `mcp__<server>__<tool>`. Pre-approve via `allowed_tools`.
Give each agent only the tools it needs.

In [ ]:
import asyncio
from typing import Any
from claude_agent_sdk import (tool, create_sdk_mcp_server, query,
    ClaudeAgentOptions, AssistantMessage, ToolUseBlock, ResultMessage)

_ORDERS = {"O-501": {"customer_id": "C-100", "total": 240.0, "status": "shipped"}}

@tool("lookup_order",
      "Look up an ORDER by order ID (O-###). Returns total, status, owning customer.",
      {"order_id": str})
async def lookup_order(args: dict[str, Any]):
    rec = _ORDERS.get(args["order_id"])
    if not rec:
        return {"content": [{"type": "text", "text": "No such order."}], "is_error": True}
    return {"content": [{"type": "text", "text": str(rec)}]}

support = create_sdk_mcp_server(name="support", version="1.0.0", tools=[lookup_order])

async def demo_tools():
    opts = ClaudeAgentOptions(mcp_servers={"support": support},
                              allowed_tools=["mcp__support__lookup_order"])
    async for m in query(prompt="What's the status of order O-501?", options=opts):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock):
                    print("[tool]", b.name, b.input)
        elif isinstance(m, ResultMessage) and m.subtype == "success":
            print("FINAL:", m.result)

# (LIVE)  In Colab: await demo_tools()
# await demo_tools()

## 3. Multi-agent orchestration & subagent delegation
A coordinator delegates to subagents defined with `AgentDefinition`. The coordinator
needs the spawn tool in `allowed_tools` — historically **"Task"**, renamed to
**"Agent"** in Claude Code v2.1.63. Include both for compatibility. Subagents have
**isolated context** and cannot spawn their own subagents.

In [ ]:
from claude_agent_sdk import AgentDefinition

def research_options():
    return ClaudeAgentOptions(
        allowed_tools=["WebSearch", "Read", "Task", "Agent"],  # include BOTH spawn names
        system_prompt=("You are a research COORDINATOR. Decompose the topic into "
                       "BROAD non-overlapping subtopics covering the WHOLE question. "
                       "Spawn parallel researchers; require source URL + date."),
        agents={
            "web-researcher": AgentDefinition(
                description="Searches one subtopic; returns cited findings.",
                prompt="Research ONE subtopic. Return claim+excerpt+URL+date per finding.",
                tools=["WebSearch", "Read"], model="sonnet"),
            "synthesizer": AgentDefinition(
                description="Combines findings into a cited report.",
                prompt="Synthesize findings passed in the prompt; preserve sources; flag gaps.",
                tools=["Read"], model="sonnet"),
        })

async def demo_research():
    async for m in query(
        prompt="Impact of AI on creative industries (music, writing, film, visual art).",
        options=research_options()):
        if hasattr(m, "content") and m.content:
            for b in m.content:
                if getattr(b, "type", None) == "tool_use" and b.name in ("Task", "Agent"):
                    print("[spawn]", b.input.get("subagent_type"))
        if hasattr(m, "result"):
            print("REPORT:", m.result)

# (LIVE)  await demo_research()

## 4. Lifecycle hooks — deterministic control & normalization
`PreToolUse` can allow/deny/modify a call **before** it runs (use for hard business
rules). `PostToolUse` can normalize/augment results **after** (use for heterogeneous
data formats). Hooks beat prompts when compliance must be guaranteed.

In [ ]:
import datetime as dt, json
from claude_agent_sdk import ClaudeSDKClient, HookMatcher

REFUND_LIMIT = 500.0

@tool("process_refund", "Refund an order. Input: order_id, amount.",
      {"order_id": str, "amount": float})
async def process_refund(args):
    return {"content": [{"type": "text", "text": f"Refunded ${args['amount']}."}]}

ops = create_sdk_mcp_server(name="ops", version="1.0.0", tools=[process_refund])

async def enforce_limit(input_data, tool_use_id, context):
    if input_data.get("tool_name") == "mcp__ops__process_refund":
        if float(input_data["tool_input"].get("amount", 0)) > REFUND_LIMIT:
            return {"hookSpecificOutput": {
                "hookEventName": input_data["hook_event_name"],
                "permissionDecision": "deny",
                "permissionDecisionReason": f"Refunds over ${REFUND_LIMIT} need a human."}}
    return {}

async def demo_hooks():
    opts = ClaudeAgentOptions(
        mcp_servers={"ops": ops},
        allowed_tools=["mcp__ops__process_refund"],
        hooks={"PreToolUse": [HookMatcher(hooks=[enforce_limit])]})
    async with ClaudeSDKClient(options=opts) as c:
        await c.query("Refund order O-501 in full ($720).")  # blocked by hook
        async for m in c.receive_response():
            if isinstance(m, (AssistantMessage, ResultMessage)):
                print(m)

# (LIVE)  await demo_hooks()

## 5. Programmatic prerequisite gate (sample Question 1)
Block `lookup_order` / `process_refund` until `get_customer` has verified identity.
State lives **outside** the model → deterministic guarantee a prompt cannot give.
This is why the exam's correct answer is the programmatic prerequisite, not a
stronger prompt or few-shot examples.

In [ ]:
VERIFIED = set()
GATED = {"mcp__support__lookup_order", "mcp__support__process_refund"}

async def require_verification(input_data, tool_use_id, context):
    if input_data.get("tool_name") in GATED and not VERIFIED:
        return {"hookSpecificOutput": {
            "hookEventName": input_data["hook_event_name"],
            "permissionDecision": "deny",
            "permissionDecisionReason": "Call get_customer to verify identity first."}}
    return {}

print("Gate ready. Wire require_verification as a PreToolUse hook (see 05_*.py).")

---
### Exam recap
- Loop on `stop_reason`; append tool results as a user turn.
- Scope tools per agent; namespaced `mcp__server__tool`; `is_error` for failures.
- Coordinator needs Task/Agent; subagents = isolated context, no nested spawning.
- Hooks = deterministic; prompts = probabilistic. Use hooks for must-hold rules.
- Prerequisite gates enforce ordering deterministically.